# BinSense — M4: Train the Detector (YOLOv8, 1 class)

Train a single-class (`item`) object detector on the 120-bin manual seed, then
evaluate it two ways:
- **Detection mAP@50** on the held-out *val split of labeled bins* (needs boxes).
- **Total-count accuracy (±1)** on the *eval-gold* bins vs `EXPECTED_QUANTITY`
  (needs only a number — no boxes).

> **Model decision (recorded for the write-up/video): YOLOv8s.**
> The labeled seed is small (~120 bins, one class) and bins are dense/occluded. A
> **small COCO-pretrained model + heavy augmentation** avoids overfitting, trains
> fast on a Colab T4, and gives clean Ultralytics `val` metrics. YOLOv8n (faster,
> slightly weaker) and YOLOv8m (stronger but overfit-prone here) are one line away
> via `MODEL`. YOLOv11s offers marginal gains with less reference coverage.
> **Colab GPU is essential** — training is thousands of forward/backward passes;
> a run that's hours on CPU is minutes on a T4.

In [ ]:
# Cell 1: Bootstrap — path resolution + data/code split (+ ultralytics on Colab)
import sys, os, subprocess
from pathlib import Path

GITHUB_URL = 'https://github.com/rishib09/AmazonBinSense.git'
BRANCH     = 'm4-train-yolo'   # milestone branch holding this notebook's code (set 'master' after merge)
DRIVE_ROOT = '/content/drive/MyDrive/Interview Kickstart/Capstone Project/Amazon BinSense'
LOCAL_DATA = r'G:\My Drive\Interview Kickstart\Capstone Project\Amazon BinSense\data'

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/AmazonBinSense')
    if (PROJECT_ROOT / '.git').exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH], check=False)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH], check=False)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', 'origin', BRANCH, '--ff-only'], check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_URL, str(PROJECT_ROOT)], check=True)
    os.environ['BINSENSE_DATA_DIR'] = str(Path(DRIVE_ROOT) / 'data')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
except ImportError:
    IN_COLAB = False
    if os.getenv('BINSENSE_DIR'):
        PROJECT_ROOT = Path(os.environ['BINSENSE_DIR'])
    else:
        _cwd = Path.cwd()
        PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
    if not os.getenv('BINSENSE_DATA_DIR') and Path(LOCAL_DATA).exists():
        os.environ['BINSENSE_DATA_DIR'] = LOCAL_DATA

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Running in:', 'Google Colab' if IN_COLAB else 'Local', '| ROOT:', PROJECT_ROOT)

In [ ]:
# Cell 2: Imports + build/verify the YOLO dataset (train/val), assert no eval leakage
import yaml, random
import pandas as pd
from utils.env_utils import setup_env, get_device, print_gpu_info

cfg = setup_env(verbose=True)
print_gpu_info()

EVAL_IDS = set(pd.read_csv(cfg.splits_dir / 'eval.csv')['bin_id'].astype(str).str.zfill(5))

labeled = sorted(p.stem for p in cfg.labels_dir.glob('*.txt'))
labeled = [b for b in labeled if b not in EVAL_IDS]              # never train on eval
assert not (set(labeled) & EVAL_IDS), 'EVAL LEAKAGE in training labels!'
assert labeled, 'No labels in data/labels/ — run M3 (ingest_labels.py) first.'

random.Random(42).shuffle(labeled)
n_val = max(1, int(0.15 * len(labeled)))
val, train = labeled[:n_val], labeled[n_val:]

YOLO_DIR = cfg.base_dir / 'data' / 'yolo'; YOLO_DIR.mkdir(parents=True, exist_ok=True)
for name, ids in [('train.txt', train), ('val.txt', val)]:
    (YOLO_DIR / name).write_text('\n'.join(str(cfg.images_dir / f'{b}.jpg') for b in ids))
DATA_YAML = YOLO_DIR / 'data.yaml'
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(cfg.data_dir), 'train': str(YOLO_DIR / 'train.txt'),
    'val': str(YOLO_DIR / 'val.txt'), 'nc': 1, 'names': {0: 'item'},
}, sort_keys=False))
print(f'train={len(train)}  val={len(val)}  eval(held out)={len(EVAL_IDS)}')
print('wrote', DATA_YAML)

## 1. Dataset sanity check
Confirm boxes land on items before spending a training run (the #1 pre-train check).

In [ ]:
from tools.labeling.overlay_check import summarize, draw_overlay, parse_label
rows = summarize(cfg.labels_dir, cfg.images_dir, cfg.metadata_dir)
import matplotlib.pyplot as plt, cv2, numpy as np
sample = [r['bin_id'] for r in rows if r['img_exists']][:3]
fig, axes = plt.subplots(1, len(sample), figsize=(5*len(sample), 5))
for ax, bid in zip(np.atleast_1d(axes), sample):
    out = cfg.base_dir / 'reports' / 'label_overlays' / f'{bid}.jpg'
    draw_overlay(cfg.images_dir/f'{bid}.jpg', parse_label(cfg.labels_dir/f'{bid}.txt'), out)
    ax.imshow(cv2.cvtColor(cv2.imread(str(out)), cv2.COLOR_BGR2RGB)); ax.set_title(bid); ax.axis('off')
plt.suptitle('Label sanity — boxes must hug items'); plt.show()

## 2. Train (YOLOv8s)

**Config rationale (recorded):**
- `MODEL='yolov8s.pt'` — small, COCO-pretrained (transfer learning).
- `epochs=100`, `patience=20` — early-stop; the small set converges fast.
- `imgsz=640` — standard; bins are ~470px so 640 upsamples slightly (fine).
- `batch=16` — fits a T4; lower if OOM.
- Augmentation ON (mosaic, HSV, flips) — critical on a ~120-image set to fight overfit.
- Runs save to `cfg.models_dir` → resolves to **Drive** (`…/Amazon BinSense/models`), so
  weights persist across Colab sessions. (If Drive I/O slows training, set `project` to a
  local dir and copy the run to `cfg.models_dir` at the end.)

In [ ]:
from ultralytics import YOLO

MODEL   = 'yolov8s.pt'   # <- swap to yolov8n.pt / yolov8m.pt / yolo11s.pt to compare
EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16

model = YOLO(MODEL)
results = model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    patience=20, seed=42, pretrained=True,
    project=str(cfg.models_dir), name='yolov8s_item', exist_ok=True,
    device=0 if get_device().type == 'cuda' else 'cpu',
)
RUN_DIR = Path(results.save_dir)
print('run dir:', RUN_DIR)

## 3. Training curves

In [ ]:
from IPython.display import Image, display
for png in ['results.png', 'PR_curve.png', 'confusion_matrix.png']:
    p = RUN_DIR / png
    if p.exists():
        print(png); display(Image(filename=str(p)))

## 4. Detection metrics (mAP@50 on the labeled val split)

In [ ]:
metrics = model.val(data=str(DATA_YAML), imgsz=IMGSZ, verbose=False)
print(f'mAP@50    : {metrics.box.map50:.3f}   (Tier-2 target >= 0.60)')
print(f'mAP@50-95 : {metrics.box.map:.3f}')
print(f'precision : {metrics.box.mp:.3f}   recall: {metrics.box.mr:.3f}')

## 5. Total-count accuracy on eval-gold (Tier-1)
Eval bins are unlabeled, so we compare `#detections` to `EXPECTED_QUANTITY`.
A negative `mean_signed_diff` is expected (occlusion → visible undercount).

In [ ]:
from tools.detect.count_eval import count_predictions, summarize_counts

eval_ids = sorted(EVAL_IDS)
rows = count_predictions(model, eval_ids, cfg.images_dir, cfg.metadata_dir, conf=0.25, imgsz=IMGSZ)
summary = summarize_counts(rows)
print('Count metrics vs EXPECTED_QUANTITY (eval-gold):')
for k, v in summary.items():
    print(f'  {k:16s}: {v}')
print('\nTier-1 target: within1_pct >= 75')
dfc = pd.DataFrame(rows)
display(dfc.sort_values('diff').head(10))   # worst undercounts (dense/occluded bins)

In [ ]:
# Sample predictions on a few eval bins
import numpy as np
show = eval_ids[:3]
fig, axes = plt.subplots(1, len(show), figsize=(5*len(show), 5))
for ax, bid in zip(np.atleast_1d(axes), show):
    res = model.predict(str(cfg.images_dir/f'{bid}.jpg'), conf=0.25, imgsz=IMGSZ, verbose=False)[0]
    ax.imshow(cv2.cvtColor(res.plot()[..., ::-1], cv2.COLOR_BGR2RGB))
    ax.set_title(f'{bid}: {len(res.boxes)} det'); ax.axis('off')
plt.suptitle('Detections on eval-gold bins'); plt.show()

## 6. Results vs targets & handoff to M5

- **mAP@50** (Tier-2 gate >= 0.60) and **count within-1** (Tier-1 gate >= 75%) —
  read the numbers above; if short, the usual levers are more labels (SAM/zero-shot
  passes in nb 03), more epochs, or a larger model (`yolov8m`).
- **Record the actual hyperparameters + metrics here** for the decision log / video.
- Best weights: `RUN_DIR/weights/best.pt` → these produce the **crops** that M5's
  embedder + FAISS gallery consume next.